# Data Cleaning: Road Accident Prediction

สมุดบันทึกนี้ใช้ฟังก์ชันทำความสะอาดข้อมูลชุดเดียวกับ pipeline หลักของโปรเจกต์ เพื่อให้ผลลัพธ์ใน notebook และสคริปต์ production ตรงกัน

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").exists():
    raise FileNotFoundError("ไม่พบโฟลเดอร์ src กรุณาเปิด notebook จาก project root หรือโฟลเดอร์ notebooks")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from IPython.display import display

from src.data.load_data import load_accident_files
from src.data.clean_data import clean_accident_data
from src.utils.config import RAW_DIR, PROCESSED_CSV, PROCESSED_PARQUET, ensure_project_dirs


In [ ]:
df_raw = load_accident_files(str(RAW_DIR))
print(f"raw shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
df_raw.head()


In [ ]:
print(f"duplicate rows: {df_raw.duplicated().sum():,}")
print(f"source files: {sorted(df_raw['source_file'].dropna().unique().tolist())}")

raw_profile = (
    pd.DataFrame(
        {
            "dtype": df_raw.dtypes.astype(str),
            "missing_count": df_raw.isna().sum(),
            "missing_pct": (df_raw.isna().mean() * 100).round(2),
        }
    )
    .sort_values(["missing_count", "dtype"], ascending=[False, True])
)

raw_profile.head(15)


In [ ]:
df_clean = clean_accident_data(df_raw)

comparison = pd.DataFrame(
    {
        "raw": [len(df_raw), df_raw.shape[1], int(df_raw.duplicated().sum())],
        "clean": [len(df_clean), df_clean.shape[1], int(df_clean.duplicated().sum())],
    },
    index=["rows", "columns", "duplicate_rows"],
)

comparison


In [ ]:
key_cols = [
    c
    for c in [
        "จังหวัด", "สภาพอากาศ", "ลักษณะการเกิดเหตุ", "มูลเหตุสันนิษฐาน",
        "บริเวณที่เกิดเหตุ", "LATITUDE", "LONGITUDE", "hour", "day_of_week",
        "month", "is_peak_hour", "รวมจำนวนผู้บาดเจ็บ"
    ]
    if c in df_clean.columns
]

post_clean_checks = pd.DataFrame(
    {
        "dtype": df_clean[key_cols].dtypes.astype(str),
        "missing_count": df_clean[key_cols].isna().sum(),
        "missing_pct": (df_clean[key_cols].isna().mean() * 100).round(2),
    }
)

display(df_clean[key_cols].head())
post_clean_checks


In [ ]:
ensure_project_dirs()
df_clean.to_csv(PROCESSED_CSV, index=False, encoding="utf-8-sig")
df_clean.to_parquet(PROCESSED_PARQUET, index=False)

print(f"saved csv: {PROCESSED_CSV}")
print(f"saved parquet: {PROCESSED_PARQUET}")


## หมายเหตุ

`src.data.run_preprocess` ใช้ logic เดียวกันกับ notebook นี้ หากต้องการสร้างไฟล์แบบ batch ให้ใช้สคริปต์ดังกล่าวได้ทันที